# Automatic Differentiation: Hands-On with Clad

[**Clad**](https://github.com/vgvassilev/clad) is a source-to-source automatic-differentiation (AD) plugin for the Clang/LLVM compiler. Given a C++ function, it *generates the C++ source of its derivative* at compile time (here, at ROOT/Cling JIT time) which is then compiled like any other code.
The derivative is therefore **exact** (not finite differences) and as fast as hand-written code.

In this notebook, we'll try out **Clad** through the [C++ interpreter in ROOT, called Cling](https://root.cern/manual/cling/), steered from the [ROOT Python Interface](https://root.cern/manual/python/).

This notebook is a hands-on tour in three sections:

1. **Clad fundamentals**: forward vs. reverse mode, inspecting the generated code, driving Clad from Python, and Hessians.
2. **Measuring the Z mass from real CMS open data**: a binned likelihood fit with ROOT's Minuit2, with gradient *and* Hessian from one function.
3. **The same fit with RooFit**: declarative shapes, automatic normalization — and the gradient generated automatically by Clad via the codegen backend.

We write the C++ with ROOT's `%%cpp` cell magic: `--declare` *defines* functions; plain `%%cpp` runs the `clad::*` call that *triggers code generation*. The generated function then appears as a normal `ROOT.<name>`, callable from Python.


## Setup

Importing `ROOT` in a Jupyter kernel registers the `%%cpp` cell magic used throughout.

> **If you hit `error: redefinition of '<name>'`:** Cling does not allow redefining a
> symbol, so re-running a `%%cpp --declare` cell fails. Recovery: **Kernel → Restart**,
> then re-run the cells above yours.


In [ ]:
import numpy as np
import ROOT
# interactive inline canvases, straight from ROOT
%jsroot on

# warm up the Cling + Clad JIT (a few seconds) so later cells are snappy
ROOT.gInterpreter.Declare("#include <Math/CladDerivator.h>")
ROOT.gInterpreter.Declare("double _warm(double z){ return z*z; }")
ROOT.gInterpreter.ProcessLine("clad::gradient(_warm);")
print("ROOT", ROOT.gROOT.GetVersion())


## 1. Clad fundamentals

Three entry points, each taking a function and returning a callable:

| call | mode | gives |
|------|------|-------|
| `clad::differentiate(f, "x")` | forward | one directional derivative |
| `clad::gradient(f)` | reverse | all first partials at once |
| `clad::hessian(f, "x,y")` | second order | the full Hessian |

Our running example is $f(x, y) = x^2 y + \sin x$, with
$\partial f/\partial x = 2xy + \cos x$, $\partial f/\partial y = x^2$, and
$\partial^2 f/\partial x^2 = 2y - \sin x$, $\partial^2 f/\partial x\,\partial y = 2x$,
$\partial^2 f/\partial y^2 = 0$.

In [ ]:
%%cpp --declare

double f(double x, double y)
{
   return x * x * y + std::sin(x);
}


In [ ]:
x = 1.3
y = 2.0

### 1.1 Forward mode: `clad::differentiate`

Forward mode pushes one input's perturbation through the computation: one generated
function per input, each call returning that partial derivative. Cost scales with the
number of **inputs**. Good for few inputs, many outputs.


In [ ]:
%%cpp
clad::differentiate(f, "x"); // -> f_darg0
clad::differentiate(f, "y"); // -> f_darg1

In [ ]:
dfdx = ROOT.f_darg0(x, y)   # forward mode RETURNS the derivative directly
dfdy = ROOT.f_darg1(x, y)
print(f"df/dx = {dfdx:.6f}   (analytic {2*x*y + np.cos(x):.6f})")
print(f"df/dy = {dfdy:.6f}   (analytic {x*x:.6f})")

### 1.2 Reverse mode: `clad::gradient`

Reverse mode propagates the output's sensitivity backwards: **one call gives all partials
at once**. Cost scales with the number of **outputs**. The mode for fitting/ML, where one
scalar loss depends on many parameters. The gradient comes back through trailing output
arguments.


In [ ]:
%%cpp
clad::gradient(f); // -> f_grad

In [ ]:
# gradients come back through trailing double* args -> pass length-1 numpy arrays
gx = np.zeros(1)
gy = np.zeros(1)
ROOT.f_grad(x, y, gx, gy)
print(f"grad f = ({gx[0]:.6f}, {gy[0]:.6f})   "
      f"(analytic ({2*x*y + np.cos(x):.6f}, {x*x:.6f}))")

### 1.3 Inspecting the generated code with `.dump()`

Each entry point returns an object whose `.dump()` prints the generated source in plain C++.
You could paste into a codebase that does not link Clad at all.


In [ ]:
%%cpp
{
   auto d = clad::differentiate(f, "x");
   d.dump();
}


In [ ]:
%%cpp
{
   auto g = clad::gradient(f);
   g.dump();
}


### 1.4 From Python: array parameters

Real models pack parameters into a `double*`; from Python, pass one numpy array for the
parameters and one to receive the gradient. Two rules that bite: **contiguous float64**
arrays, and **zero the gradient array** before each call, as Clad *accumulates* into it.


In [ ]:
%%cpp --declare

// same f as before, parameters packed into an array
double g_arr(double *p)
{
   return p[0] * p[0] * p[1] + std::sin(p[0]);
}


In [ ]:
%%cpp
clad::gradient(g_arr, "p"); // -> g_grad


In [ ]:
p = np.array([x, y])
grad = np.zeros(2)                 # must be zeroed: Clad adds into it
ROOT.g_arr_grad(p, grad)               # generated name: g_grad
print("params p  =", p)
print("grad g(p) =", grad, "  (same numbers, array-valued)")

### 1.5 Hessians with `clad::hessian`

`clad::hessian` fills a flattened $n \times n$ buffer of second derivatives
(forward-over-reverse under the hood); reshape to $(n, n)$. The headline use is
uncertainties: for a negative log-likelihood, the parameter covariance is the inverse
Hessian at the minimum, $C = H^{-1}$.


In [ ]:
%%cpp
clad::hessian(f, "x,y"); // -> f_hessian


In [ ]:
H = np.zeros(4)
ROOT.f_hessian(x, y, H)
H = H.reshape(2, 2)
print("Hessian =")
print(H)
print("analytic = [[2y - sin(x), 2x], [2x, 0]] =",
      [[round(2*y - np.sin(x), 4), 2*x], [2*x, 0.0]])

**Exercise.** Your "Hello World" with Clad: `cube(x) = x*x*x` has derivative $3x^2$,
so $12$ at $x = 2$. Use Clad to generate the derivative and check it.

*(Exercises drive Clad from Python via `ROOT.gInterpreter`, which is exactly what `%%cpp` does
under the hood, so everything fits in one cell.)*


In [ ]:
# EXERCISE: differentiate cube(x) = x*x*x with Clad and check cube'(2) == 12.
#   1. declare:         ROOT.gInterpreter.Declare("double cube(double x){ return x*x*x; }")
#   2. differentiate:   ROOT.gInterpreter.ProcessLine("clad::gradient(cube);")
#   3. call cube_grad(2.0, out) with a length-1 numpy array 'out'; print out[0]



**Solution** *(Collapsed below, expand after giving it a try! A plain "Run All" still executes it)*.


In [ ]:
ROOT.gInterpreter.Declare("double cube(double x){ return x*x*x; }")
ROOT.gInterpreter.ProcessLine("clad::gradient(cube);")
d = np.zeros(1)
ROOT.cube_grad(2.0, d)
print("cube'(2) =", d[0], " (analytic 12)")


**Exercise.** Write functions that Clad won't be able to differentiate.

In particular, can you write a function that will cause Clad to error out?

Can you write a function that Clad happily differentiates, but the derivative result is wrong? Why does this happen?

## 2. Measuring the Z mass from real CMS open data

The finale: **real data**! A proper likelihood, a real minimizer
(ROOT's Minuit2), and Hessian-based uncertainties. We're looking at an actual measurement: the Z
boson mass, and how many Z bosons are in the sample.

We model the signal with a **Gaussian**

$$S(m) = \exp\!\Big(-\frac{(m - M)^2}{2\sigma^2}\Big),$$

standing in for the Z lineshape convolved with the detector resolution, on a falling
exponential background $B(m) = e^{-\lambda m}$. A mass fit normally has to work out the
**normalization** integral of the lineshape over the fit range (an \(\mathrm{erf}\) for the
Gaussian, and already nothing elementary for e.g. a Breit-Wigner). The **binned
extended-Poisson** likelihood we'll write sidesteps analytic normalization with a plain
*sum over bins*:

$$\nu_i = N_s\,\frac{S(m_i)}{\sum_j S(m_j)} + N_b\,\frac{B(m_i)}{\sum_j B(m_j)},$$

at fixed bin centers $m_i$, with extended NLL $\sum_i (\nu_i - n_i \ln \nu_i)$. No
special functions, nothing exotic for Clad. **Note**: the normalization
$\sum_j S(m_j;\,M,\sigma)$ depends on the shape parameters, so
$\partial(\mathrm{NLL})/\partial M$ has a term flowing through it, which hand-coded
gradients routinely forget.

The data: 40 bins of 1 GeV over 70-110 GeV of opposite-sign dimuon invariant masses from
the CMS Run2011A DoubleMu dataset ([CERN Open Data record 545](https://opendata.cern.ch/record/545)), pre-binned so there is no
data loading.


In [ ]:
COUNTS = np.array([
    34, 30, 42, 30, 35, 53, 42, 43, 42, 47, 62, 59, 70, 94, 96, 128,
    171, 263, 422, 675, 806, 873, 595, 314, 228, 107, 80, 54, 46, 31,
    27, 21, 12, 14, 13, 19, 11, 7, 13, 7], dtype=float)
M_LO, M_HI, N_BINS = 70.0, 110.0, 40
BIN_W = (M_HI - M_LO) / N_BINS
CENTERS = M_LO + (np.arange(N_BINS) + 0.5) * BIN_W
PDG_MZ = 91.1876

print(f"{int(COUNTS.sum())} events; tallest bin {CENTERS[COUNTS.argmax()]:.1f} GeV with {int(COUNTS.max())} events")

h_data = ROOT.TH1F("h_data", "CMS open data: dimuon spectrum;"
                   "dimuon invariant mass [GeV];events / GeV", N_BINS, M_LO, M_HI)
for i, n in enumerate(COUNTS):
    h_data.SetBinContent(i + 1, n)
h_data.SetFillColor(ROOT.kAzure - 8)
h_data.SetBarWidth(0.9)
h_data.SetStats(False)  # binned counts: the stats box would be misleading
c_data = ROOT.TCanvas("c_data", "CMS open data", 800, 480)
h_data.Draw("BAR")
c_data.Draw()

### Where the counts come from (reproducing them)

`COUNTS` is real CMS data, pre-binned so the notebook needs no network or file access: the
**CMS Run2011A DoubleMu** dataset on the [CERN Open Data Portal, record 545](https://opendata.cern.ch/record/545),
whose `Dimuon_DoubleMu.csv` carries a precomputed dimuon invariant-mass column `M`
(475k events). The counts were produced once with:

```python
import numpy as np

# Dimuon_DoubleMu.csv from record 545, e.g. via XRootD:
#   root://eospublic.cern.ch//eos/opendata/cms/Run2011A/DoubleMu/CSV/12Oct2013-v1/Dimuon_DoubleMu.csv

d = np.genfromtxt("Dimuon_DoubleMu.csv", delimiter=",", names=True)
opp = d["Q1"] * d["Q2"] < 0                          # opposite-sign muons only
COUNTS, edges = np.histogram(d["M"][opp], bins=40, range=(70, 110))
```

Change `bins`/`range` and everything downstream (fit, Hessian errors) adapts.


### 2.1 The model and likelihood

The parameters `[N_s, N_b, M, sigma, lambda]` are packed into a `double*` (the array shape
from section 1), and the bin counts come in as a `const double*`. The model is written
with small line-shape functions, and an NLL that calls them in a loop. Clad
differentiates through **nested function calls** without any special treatment. From the
one NLL we generate both the gradient (for the fit) and the Hessian (for the error bars).


In [ ]:
%%cpp --declare

const int NBINS = 40;
const double MLO = 70.0;
const double MHI = 110.0;
// line shapes, bare (the NLL normalizes them by their sum over bins)
double gaussian(double m, double M, double sigma)
{
   double d = (m - M) / sigma;
   return std::exp(-0.5 * d * d);
}
double background(double m, double lam)
{
   return std::exp(-lam * m);
}
double nll(double *p, double const *n)
{
   const double binw = (MHI - MLO) / NBINS;
   double Ns = p[0], Nb = p[1], M = p[2], sigma = p[3], lam = p[4];
   double sumS = 0.0, sumB = 0.0;
   for (int i = 0; i < NBINS; ++i) {
      double m = MLO + (i + 0.5) * binw;
      sumS += gaussian(m, M, sigma);
      sumB += background(m, lam);
   }
   double val = 0.0;
   for (int i = 0; i < NBINS; ++i) {
      double m = MLO + (i + 0.5) * binw;
      double nu = Ns * gaussian(m, M, sigma) / sumS + Nb * background(m, lam) / sumB;
      val += nu - n[i] * std::log(nu);
   }
   return val;
}
// the observed counts mirrored into C++, so the Minuit2 FCN in 2.2 can reach them
double g_counts[NBINS];
void set_count(int i, double c) { g_counts[i] = c; }


**Naming of the generated functions.** Reverse mode is `<f>_grad`, the Hessian
`<f>_hessian`, forward mode `<f>_darg<k>` (one per input `k`). One wrinkle: when the
differentiated argument is an **array** *and* the function takes further arguments (as
`nll` does, taking the parameters **and** the bin counts), Clad appends an index:
`nll_grad_0` and `nll_hessian_0`. Plain-scalar functions and a lone array argument keep
the bare name.


In [ ]:
%%cpp
clad::gradient(nll, "p");     // -> nll_grad_0
clad::hessian(nll, "p[0:4]"); // -> nll_hessian_0

### 2.2 The fit with Minuit2 driven by the Clad gradient

First spot-check the fresh gradient against a central finite difference — a cheap sanity
check worth doing once for any hand-built likelihood — then fit. No parameter rescaling: the scales
differ wildly (yields ~1e3, $m_Z$ ~ 90, $\lambda$ ~ 0.03), but Minuit2 keeps a per-parameter
step size, so it fits the **natural** parameters directly. The C++ NLL is the cost; the Clad
gradient is handed to Migrad through a `ROOT::Math::GradFunctor`. The extended fit's total
prediction matches the observed count.


In [ ]:
# mirror the counts into C++, so the FCN in the next cell can reach them
for i, n in enumerate(COUNTS):
    ROOT.set_count(i, float(n))

# spot-check the fresh gradient against finite differences
p0 = np.array([4300.0, 1400.0, 91.0, 2.0, 0.03])
ga = np.zeros(5); ROOT.nll_grad_0(p0, COUNTS, ga)
gn = np.zeros(5)
for i in range(5):
    h = 1e-6 * max(1.0, abs(p0[i]))
    pp = p0.copy(); pp[i] += h
    pm = p0.copy(); pm[i] -= h
    gn[i] = (ROOT.nll(pp, COUNTS) - ROOT.nll(pm, COUNTS)) / (2*h)
print("gradient check vs finite differences:", "PASS" if np.allclose(ga, gn, rtol=1e-3) else "FAIL")


In [ ]:
%%cpp --declare

// Minuit2 through the ROOT::Math minimizer interface: a GradFunctor hands the
// Clad-generated analytic gradient straight to Migrad. GradFunctor asks for one
// partial at a time, so nll_partial recomputes the (cheap) full gradient and
// returns component i.
#include <Math/Factory.h>
#include <Math/Functor.h>

double nll_partial(const double *p, unsigned int i)
{
   double g[5] = {};
   nll_grad_0(const_cast<double *>(p), g_counts, g);
   return g[i];
}

double g_fit[5], g_err[5], g_nllmin;

int fit_zmass(double *start)
{
   std::function<double(const double *)> val =
      [](const double *p) { return nll(const_cast<double *>(p), g_counts); };
   std::function<double(const double *, unsigned int)> partial = nll_partial;
   ROOT::Math::GradFunctor fgrad(val, partial, 5);

   auto mn = std::unique_ptr<ROOT::Math::Minimizer>(
      ROOT::Math::Factory::CreateMinimizer("Minuit2", "Migrad"));
   mn->SetFunction(fgrad);
   mn->SetErrorDef(0.5); // negative log-likelihood: 1 sigma at delta-NLL = 0.5
   mn->SetLowerLimitedVariable(0, "N_s", start[0], 10.0, 1.0);
   mn->SetLowerLimitedVariable(1, "N_b", start[1], 10.0, 1.0);
   mn->SetLimitedVariable(2, "M", start[2], 0.05, 85.0, 97.0);
   mn->SetLimitedVariable(3, "sigma", start[3], 0.05, 0.5, 12.0);
   mn->SetLimitedVariable(4, "lambda", start[4], 5e-4, 1e-3, 0.2);
   mn->Minimize(); // Migrad, driven by the Clad gradient
   mn->Hesse();    // Minuit2's own numeric Hessian, for the cross-check in 2.3
   g_nllmin = mn->MinValue();
   for (int i = 0; i < 5; ++i) {
      g_fit[i] = mn->X()[i];
      g_err[i] = mn->Errors()[i];
   }
   return mn->Status();
}

double fit_value(int i) { return g_fit[i]; }
double fit_error(int i) { return g_err[i]; }
double fit_nll() { return g_nllmin; }


In [ ]:
istat = ROOT.fit_zmass(p0)   # Minuit2 / Migrad, driven by the Clad gradient
fit = np.array([ROOT.fit_value(i) for i in range(5)])
Ns, Nb, MZ, Sig, lam = fit
print(f"converged: {istat == 0}   (NLL = {ROOT.fit_nll():.2f})")
print(f"  N_s = {Ns:8.1f}   N_b = {Nb:8.1f}   (sum {Ns+Nb:.0f}, data {COUNTS.sum():.0f})")
print(f"  m_Z = {MZ:.4f} GeV   sigma = {Sig:.4f} GeV   lambda = {lam:.4f} /GeV")


### 2.3 Uncertainties by inverting the AD Hessian

For a plain NLL the covariance is exactly the inverse Hessian: $C = H^{-1}$ at the
minimum; 1-sigma errors are the square roots of its diagonal.

**Exercise.** `clad::hessian` already generated `nll_hessian_0`. Fill the 5×5 Hessian at
the fitted minimum, invert it, and read off the 1-sigma error on $m_Z$ (parameter
index 2). Store the error vector as `err` (the plot in the next section uses it). Cross-check against
Minuit2's HESSE errors from the fit cell above (`ROOT.fit_error(i)`).


In [ ]:
# EXERCISE: uncertainties from the Clad Hessian.
#   1. H = np.zeros(25); ROOT.nll_hessian_0(np.ascontiguousarray(fit), COUNTS, H)
#   2. err = np.sqrt(np.diag(np.linalg.inv(H.reshape(5, 5))))
#   3. print m_Z = fit[2] +/- err[2]; cross-check with ROOT.fit_error(i) (Minuit2 HESSE)
# (define err -- the 2.4 plot uses it)


**Solution** *(Collapsed below, expand after giving it a try! A plain "Run All" still executes it)*.


In [ ]:
H = np.zeros(25)
ROOT.nll_hessian_0(np.ascontiguousarray(fit), COUNTS, H)
cov = np.linalg.inv(H.reshape(5, 5))
err = np.sqrt(np.diag(cov))
mn_err = np.array([ROOT.fit_error(i) for i in range(5)])   # Minuit2's HESSE, as an independent cross-check

print(f"{'param':6} {'value':>12} {'Clad Hessian':>14} {'Minuit2 HESSE':>14}")
for name, vv, ee, me in zip(["N_s", "N_b", "M", "sigma", "lambda"], fit, err, mn_err):
    print(f"{name:6} {vv:12.4f} {ee:14.4f} {me:14.4f}")
print()
print(f"RESULT:  m_Z = {MZ:.3f} +/- {err[2]:.3f} GeV (stat.)   PDG: {PDG_MZ:.3f} GeV")
print(f"         N_Z = {Ns:.0f} +/- {err[0]:.0f} reconstructed Z -> mu mu decays")


### 2.4 The fit


In [ ]:
def sig_bkg(par):
    Ns, Nb, M, sigma, lam = par
    S = np.exp(-0.5 * ((CENTERS - M) / sigma)**2)
    B = np.exp(-lam * CENTERS)
    return Ns * S / S.sum(), Nb * B / B.sum()

sig, bkg = sig_bkg(fit)

g_data = ROOT.TGraphErrors(N_BINS, CENTERS, COUNTS, np.zeros(N_BINS), np.sqrt(COUNTS))
g_fit  = ROOT.TGraph(N_BINS, CENTERS, sig + bkg)
g_bkg  = ROOT.TGraph(N_BINS, CENTERS, bkg)
g_data.SetTitle(f"Z peak from CMS open data: m_Z = {MZ:.2f} +/- {err[2]:.2f} GeV (stat.);"
                "dimuon invariant mass [GeV];events / GeV")
g_data.SetMarkerStyle(20)
g_data.SetMarkerSize(0.8)
g_fit.SetLineColor(ROOT.kRed + 1)      # crimson
g_fit.SetLineWidth(2)
g_bkg.SetLineColor(ROOT.kAzure + 2)    # steel blue
g_bkg.SetLineStyle(2)

c_fit = ROOT.TCanvas("c_fit", "Z peak", 800, 550)
g_data.Draw("AP")
g_fit.Draw("L")
g_bkg.Draw("L")
l_pdg = ROOT.TLine(PDG_MZ, 0.0, PDG_MZ, COUNTS.max() * 1.05)
l_pdg.SetLineColor(ROOT.kGray + 2)
l_pdg.SetLineStyle(3)
l_pdg.Draw()
leg = ROOT.TLegend(0.55, 0.70, 0.88, 0.88)
leg.AddEntry(g_data, "CMS open data", "pe")
leg.AddEntry(g_fit, f"Gaussian fit (m_Z = {MZ:.2f} GeV)", "l")
leg.AddEntry(g_bkg, "background", "l")
leg.AddEntry(l_pdg, "PDG m_Z", "l")
leg.Draw()
c_fit.Draw()


## 3. The same fit with RooFit

Same physics question, same binned data, same line shapes — now declared to [RooFit](https://root.cern/manual/roofit/), ROOT's fitting toolkit, which takes over the machinery: probability normalization, the binned extended likelihood, and the Minuit2 minimization.

The ingredients mirror section 2 one-to-one: `mass` is the observable; a `RooDataHist` imports the binned counts straight from the `TH1` we built there; the Gaussian signal is RooFit's built-in `RooGaussian`, normalized **analytically** by construction; and the background is the same `exp(-lambda*m)` formula, handed to `RooGenericPdf` and normalized numerically by RooFit (benign for a smooth exponential).

With `EvalBackend="codegen"`, RooFit takes the automation one step further and hands the likelihood to **Clad**: it JIT-compiles the model to C++ and generates the fit gradient with the AD plugin from sections 1–2. So compared to section 2, nothing is hand-written anymore: no NLL, no gradient, no normalization — and it still lands on the same minimum, as the comparison below shows.


In [ ]:
# the same model, declared to RooFit: observable, shape parameters, yields
mass = ROOT.RooRealVar("mass", "dimuon invariant mass", M_LO, M_HI, "GeV")
MZr  = ROOT.RooRealVar("MZ", "m_Z", 91.0, 85.0, 97.0, "GeV")
sigr = ROOT.RooRealVar("sigm", "sigma", 2.0, 0.5, 12.0, "GeV")
lamr = ROOT.RooRealVar("lam", "lambda", 0.03, 1e-3, 0.2, "1/GeV")
Ns_r = ROOT.RooRealVar("Ns", "N_s", 4300.0, 1.0, 100000.0)
Nb_r = ROOT.RooRealVar("Nb", "N_b", 1400.0, 1.0, 100000.0)

sig_rf = ROOT.RooGaussian("sig", "Z signal", mass, MZr, sigr)   # analytic normalization
bkg_rf = ROOT.RooGenericPdf("bkg", "exp(-lam*mass)", ROOT.RooArgList(mass, lamr))
model_rf = ROOT.RooAddPdf("model", "Gaussian + exponential",
                          ROOT.RooArgList(sig_rf, bkg_rf), ROOT.RooArgList(Ns_r, Nb_r))

# the binned counts of section 2, straight from the histogram we built there
dh = ROOT.RooDataHist("dh", "dimuon counts", ROOT.RooArgList(mass), h_data)

# EvalBackend="codegen": RooFit JIT-compiles the model and has Clad generate
# the fit gradient automatically
res = model_rf.fitTo(dh, Save=True, PrintLevel=-1, PrintEvalErrors=-1, EvalBackend="codegen")

rf_val = np.array([v.getVal() for v in (Ns_r, Nb_r, MZr, sigr, lamr)])
rf_err = np.array([v.getError() for v in (Ns_r, Nb_r, MZr, sigr, lamr)])
print(f"fit status {res.status()} (0 = converged), covariance quality {res.covQual()} (3 = accurate)")
print(f"{'param':8}{'sec 2 (hand NLL)':>18}{'RooFit/codegen':>20}{'sec 2 err':>12}{'RooFit err':>13}")
for name, v2, vr, e2, er in zip(["N_s", "N_b", "M", "sigma", "lambda"], fit, rf_val, err, rf_err):
    print(f"{name:8}{v2:18.4f}{vr:20.4f}{e2:12.4f}{er:13.4f}")


In [ ]:
frame = mass.frame(Title=f"RooFit: m_Z = {rf_val[2]:.2f} +/- {rf_err[2]:.2f} GeV (stat.)")
frame.SetXTitle("dimuon invariant mass [GeV]")
frame.SetYTitle("events / GeV")
dh.plotOn(frame, Name="dh_pts", XErrorSize=0.0)
model_rf.plotOn(frame, Name="model_curve", LineColor=ROOT.kRed + 1, LineWidth=2)
model_rf.plotOn(frame, Name="bkg_curve", Components="bkg",
                LineColor=ROOT.kAzure + 2, LineStyle=ROOT.kDashed)
c_rf = ROOT.TCanvas("c_rf", "RooFit fit", 800, 550)
frame.Draw()
l_pdg2 = ROOT.TLine(PDG_MZ, 0.0, PDG_MZ, frame.GetMaximum() * 0.95)
l_pdg2.SetLineColor(ROOT.kGray + 2)
l_pdg2.SetLineStyle(3)
l_pdg2.Draw()
leg_rf = ROOT.TLegend(0.55, 0.70, 0.88, 0.88)
leg_rf.AddEntry(frame.findObject("dh_pts"), "CMS open data", "pe")
leg_rf.AddEntry(frame.findObject("model_curve"), "Gaussian fit", "l")
leg_rf.AddEntry(frame.findObject("bkg_curve"), "background", "l")
leg_rf.AddEntry(l_pdg2, "PDG m_Z", "l")
leg_rf.Draw()
c_rf.Draw()


## Conclusions

Two ingredients did all the work, both generated from ordinary C++ at Cling JIT time:

- **exact first derivatives**, forward vs. reverse mode and when each wins;
- **Hessians** for uncertainties ($C = H^{-1}$), cross-checked against Minuit2's HESSE;
- and the same likelihood re-expressed declaratively in **RooFit**, landing on the same numbers.

We ended with a real measurement: the Z mass and yield from CMS open data, fit with ROOT's
Minuit2 driven by the Clad gradient, with the statistical error straight from the inverse
of the Clad Hessian.

The short demo here was just the tip of the iceberg!
Automatic Differentiation (AD) can be used for many applications, and there are many tools that enable AD for different programming languages in various ways.

For Clad, the compelling story is: you do not rewrite your physics into a tensor
framework to get gradients. **If you can write it in C++, you can differentiate it**,
and plug the result straight into an optimizer, a sampler, or an error propagation.
